<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

# Sort Tuesday scenes — BLURRY post-2023 batch (button version — no Shift+Enter trap)

Sorts the post-blur Tuesdays (2023-08-15 → 2024-12-31) in **two batches**:
**`tuesdays_2023_gap`** (160 sheets) first, then **`tuesdays_2024`** (423 sheets).
To switch batches, change the one `SESSION_DIR` line in the first code cell, then
**Kernel → Restart Kernel** and run both cells again. **Kernel: `scaleworm`.**

**How to use:** run the two code cells below **once** (top to bottom). A panel of buttons appears with the first contact sheet and a 💡 Monday hint.

- **✓ Scene 1** — files it as Scene-1 at the time in the `scene1 t(s)` box (pre-filled from the hint; change it if the real onset differs, then click).
- **✗ Not Scene 1** — the camera is not showing the vent (wrong view / framing).
- **✗ Unusable (blur)** — the vent may be there but the image is too blurry to read. Filed as `unusable_blur` = **missing data (NA)**, kept separate from "Not Scene 1" and never counted as a zero.
- **↶ Undo** — send the last one back and re-decide.

**Every button now saves a permanent, logged decision** (there is no "Skip"). The old Skip only lived for the session and was lost on close — that is why the earlier blurry batches had to be re-recorded. If you truly want to defer a sheet, just leave it and close; it stays pending.

You **never re-run a cell** — just click buttons. That's the whole point: the old version broke because re-running a cell with Shift+Enter kept adding cells until the file hit 186 MB. This one holds a single image no matter what.

The hint is a starting guess (Monday agreed ~69% of the time) — always confirm against the sheet. If the buttons ever stop responding, use the Jupyter menu **Kernel → Restart Kernel**, then re-run both cells.

**About these frames:** they're post-blur — softer / lower-contrast than the 2021–2023 clear window. The worms are still present, so judge the front-on Mushroom view the same way; the blur is dealt with later at the counting stage (v3 + box-correction), not during sorting.

</span>

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import scene_sorter as sorter
import scene_presort as presort

# --- CONFIG: the Tuesday session to sort ------------------------------------
# Blurry post-2023 Tuesdays, done in TWO batches.
# BATCH 1 (tuesdays_2023_gap) DONE. Now on BATCH 2 (tuesdays_2024).
# To go back and finish the 6 leftover gap sheets, swap the comments below.
# SESSION_DIR = Path.cwd().parent / "scene_sorting" / "tuesdays_2023_gap"   # batch 1
SESSION_DIR = Path.cwd().parent / "scene_sorting" / "tuesdays_2024"         # batch 2

# Hints must come ONLY from the paired-Monday sorts, never from other
# day-of-week sessions, so exclude every non-Monday session dir.
_EXCLUDE = (
    "tuesdays_2021_2024", "tuesdays_2023_gap", "tuesdays_2024",
    "validation_tuesdays", "wednesdays_2021_2024",
)
_decisions = presort.load_decisions(SESSION_DIR.parent, exclude_dirs=_EXCLUDE)
_pending_stems = [p.stem for p in sorter.pending_sheets(SESSION_DIR)]
SUGGESTIONS = presort.build_suggestions(_pending_stems, _decisions)

print("Sorting:", SESSION_DIR)
print("Status :", sorter.counts(SESSION_DIR))
print(f"Pre-seed hints available for {len(SUGGESTIONS)}/{len(_pending_stems)} pending sheets")
print("NOTE: these are POST-BLUR frames (2023-08-10+) — sheets look softer / lower-"
      "contrast than the clear window. Worms are still there; judge the front-on "
      "Mushroom view as usual (the blur is handled later at counting, not here).")


In [ ]:
# --- Button-based sorter: run once, then just click. No cell re-running. ----
import csv
import shutil

import ipywidgets as widgets
from IPython.display import clear_output, display


class TuesdaySorter:
    """Sort Scene-1 / Not / Unusable-blur by clicking buttons; sheet redraws in place.

    Nothing here needs re-running, so the notebook can't accumulate cells/outputs.
    The Output area is cleared on every redraw, so it holds exactly one image.

    Every button is a TERMINAL, LOGGED decision -- there is no in-session-only
    "skip" any more (the old Skip discarded the judgment when the notebook closed).
    "Unusable (blur)" files the sheet as unusable_blur = missing data / NA, kept
    DISTINCT from "Not Scene 1" = the camera was not showing the vent.
    """

    def __init__(self):
        self.last_t = 210  # carry-forward default when a frame has no hint time

        self.status = widgets.HTML()
        self.hint = widgets.HTML()
        self.out = widgets.Output()
        self.msg = widgets.Output()
        self.time_box = widgets.IntText(
            value=self.last_t, description="scene1 t(s):",
            layout=widgets.Layout(width="200px"),
        )

        def mk(desc, style=""):
            return widgets.Button(
                description=desc, button_style=style,
                layout=widgets.Layout(width="auto"),
            )

        self.b_s1 = mk("✓ Scene 1", "success")
        self.b_no = mk("✗ Not Scene 1", "warning")
        self.b_blur = mk("✗ Unusable (blur)", "danger")
        self.b_undo = mk("↶ Undo")
        self.b_s1.on_click(lambda _: self._decide("scene1", self.time_box.value))
        self.b_no.on_click(lambda _: self._decide("not_scene1"))
        self.b_blur.on_click(lambda _: self._decide("unusable_blur"))
        self.b_undo.on_click(lambda _: self._undo())

        controls = widgets.HBox(
            [self.b_s1, self.time_box, self.b_no, self.b_blur, self.b_undo]
        )
        display(widgets.VBox([self.status, self.hint, controls, self.out, self.msg]))
        self._draw()

    def _pending(self):
        return sorter.pending_sheets(SESSION_DIR)

    def _draw(self):
        rem = self._pending()
        c = sorter.counts(SESSION_DIR)
        self.status.value = (
            f"<b>Scene 1:</b> {c['scene1']} &nbsp;|&nbsp; "
            f"<b>Not:</b> {c['not_scene1']} &nbsp;|&nbsp; "
            f"<b>Blur:</b> {c['unusable_blur']} &nbsp;|&nbsp; "
            f"<b>remaining:</b> {len(rem)}"
        )
        with self.out:
            clear_output(wait=True)
            if not rem:
                self.hint.value = "<b>✅ Nothing left to sort in this session.</b>"
                return
            sheet = rem[0]
            sug = SUGGESTIONS.get(sheet.stem)
            if sug:
                self.hint.value = sug.hint()
                if sug.decision == "scene1" and sug.scene1_time_s is not None:
                    self.time_box.value = int(sug.scene1_time_s)
                else:
                    self.time_box.value = self.last_t
            else:
                self.hint.value = "ℹ no paired-Monday hint — sort this one cold"
                self.time_box.value = self.last_t
            fig, ax = plt.subplots(figsize=(15, 8))
            ax.imshow(plt.imread(sheet))
            ax.axis("off")
            ax.set_title(sheet.stem, fontsize=11)
            plt.show()
            plt.close(fig)

    def _decide(self, decision, t=None):
        rem = self._pending()
        if not rem:
            self._draw()
            return
        sheet = rem[0]
        scene_t = None
        if decision == "scene1":
            scene_t = float(t)
            self.last_t = int(t)
        sorter.apply_decision(SESSION_DIR, sheet, decision, scene1_time_s=scene_t)
        self._draw()

    def _undo(self):
        paths = sorter.session_paths(SESSION_DIR)
        with self.msg:
            clear_output(wait=True)
            if not paths.log_csv.exists():
                print("Nothing to undo yet.")
                return
            rows = list(csv.DictReader(paths.log_csv.open()))
            filed = [r for r in rows if r["decision"] in sorter.FILED_DECISIONS]
            if not filed:
                print("Nothing filed to undo.")
                return
            last = filed[-1]
            name = last["stem"] + ".png"
            src = paths.target_for(last["decision"]) / name
            if src.is_file():
                shutil.move(str(src), str(paths.contact_sheets / name))
            # drop the last log row for that stem so it doesn't double-count
            keep, dropped = [], False
            for r in reversed(rows):
                if (not dropped and r["stem"] == last["stem"]
                        and r["decision"] == last["decision"]):
                    dropped = True
                    continue
                keep.append(r)
            keep.reverse()
            with paths.log_csv.open("w", newline="") as fh:
                w = csv.DictWriter(fh, fieldnames=list(sorter.LOG_HEADER))
                w.writeheader()
                w.writerows(keep)
            print(f"Undid {name} — it's back in the queue to re-decide.")
        self._draw()


SORTER = TuesdaySorter()
